In [1]:
import pandas as pd
import numpy as np

PATH = "../data/raw/Most-Recent-Cohorts-Institution.csv"

# The file has ~3,000 columns. We only load the ones relevant to matching.
# Keys = original Scorecard code, values = readable name we'll use.
COLS = {
    "UNITID": "unit_id",            # unique school ID (join key for other datasets)
    "INSTNM": "name",
    "CITY": "city",
    "STABBR": "state",
    "LOCALE": "locale",             # urban-centric locale code (city/suburb/town/rural x size)
    "CONTROL": "control",           # 1=public, 2=private nonprofit, 3=for-profit
    "ICLEVEL": "level",             # 1=4-year, 2=2-year (JUCO), 3=<2-year
    "CURROPER": "operating",        # 1=currently operating
    "PREDDEG": "main_degree",       # degree most students earn: 1=certificate, 2=associate, 3=bachelor's, 4=graduate
    "OPENADMP": "open_admission_policy",  # 1=open admission, 2=not open
    "UGDS": "undergrads",           # undergrad enrollment
    "UGDS_NRA": "pct_international",# share of undergrads who are international
    "ADM_RATE": "admit_rate",
    "SAT_AVG": "sat_avg",
    "COSTT4_A": "cost_per_year",    # total cost of attendance (tuition+housing+etc.)
    "TUITIONFEE_OUT": "tuition_out_of_state",  # internationals usually pay this rate
    "TUITIONFEE_IN": "tuition_in_state",
    "C150_4": "grad_rate_4yr",      # % finishing within 150% time, 4-year schools
    "C150_L4": "grad_rate_2yr",     # same, for 2-year schools
    "MD_EARN_WNE_P10": "median_earnings_10yr",
    "LATITUDE": "lat",
    "LONGITUDE": "lon",
    "INSTURL": "website",        # school website URL
}

# Read only the header first to confirm the columns exist (codes occasionally change)
header = pd.read_csv(PATH, nrows=0).columns
missing_cols = [c for c in COLS if c not in header]
print("Columns not found:", missing_cols)

# "PrivacySuppressed" = Dept. of Ed hid the value (too few students to report safely).
# Treating it as missing keeps number columns numeric.
df = pd.read_csv(
    PATH,
    usecols=[c for c in COLS if c in header],
    na_values=["PrivacySuppressed", "NULL"],
    low_memory=False,
).rename(columns=COLS)

# --- Sanity check ---
print("Shape (rows, cols):", df.shape)
df.info()   # dtypes + non-null counts
print((df.isna().mean() * 100).round(1).sort_values(ascending=False))

Columns not found: []


Shape (rows, cols): (6273, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6273 entries, 0 to 6272
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   unit_id                6273 non-null   int64  
 1   name                   6273 non-null   object 
 2   city                   6273 non-null   object 
 3   state                  6273 non-null   object 
 4   website                6263 non-null   object 
 5   main_degree            6273 non-null   int64  
 6   control                6273 non-null   int64  
 7   locale                 5732 non-null   float64
 8   lat                    5731 non-null   float64
 9   lon                    5731 non-null   float64
 10  admit_rate             1910 non-null   float64
 11  sat_avg                1037 non-null   float64
 12  undergrads             5478 non-null   float64
 13  pct_international      5478 non-null   float64
 14  operating              62

In [2]:
# Summary stats: look for impossible values (negatives, rates > 1)
df.describe().T

,count,mean,std,min,25%,50%,75%,max
unit_id,6273.0,2.247211e+06,7.817361e+06,100654.000000,174321.000000,229504.000000,459851.000000,4.966450e+07
main_degree,6273.0,1.826877e+00,1.079581e+00,0.000000,1.000000,2.000000,3.000000,4.000000e+00
control,6273.0,2.044317e+00,8.337278e-01,1.000000,1.000000,2.000000,3.000000,3.000000e+00
locale,5732.0,1.998395e+01,9.872990e+00,-3.000000,12.000000,21.000000,22.000000,4.300000e+01
lat,5731.0,3.728396e+01,5.852503e+00,-14.322636,33.911324,38.603173,41.234638,7.132470e+01
lon,5731.0,-9.047510e+01,1.820276e+01,-170.742774,-97.546397,-86.392369,-78.935882,1.713781e+02
admit_rate,1910.0,7.275854e-01,2.312871e-01,0.000000,0.612150,0.779100,0.905275,1.000000e+00
sat_avg,1037.0,1.184465e+03,1.485244e+02,720.000000,1084.000000,1161.000000,1271.000000,1.560000e+03
undergrads,5478.0,2.641150e+03,6.526872e+03,0.000000,127.000000,563.500000,2219.750000,1.631640e+05
pct_international,5478.0,2.367023e-02,6.401427e-02,0.000000,0.000000,0.001100,0.021675,1.000000e+00


In [3]:
# Step 2: narrow to schools a Brazilian student would realistically consider.
# Print the row count after each filter so we can see how much each rule removes.
print("Start:", len(df))

schools = df[df["operating"] == 1]
print("After operating == 1:", len(schools))

schools = schools[schools["level"].isin([1, 2])]          # 1 = 4-year, 2 = 2-year (JUCO)
print("After keeping 2-year + 4-year:", len(schools))

schools = schools[schools["control"].isin([1, 2])]        # 1 = public, 2 = private nonprofit
print("After dropping for-profits:", len(schools))

schools = schools[schools["undergrads"] > 0]              # NaN > 0 is False, so missing enrollment is dropped too
print("After undergrads > 0:", len(schools))

schools = schools.copy()   # clean copy so later edits don't trigger pandas warnings

# How many of each type are left? (margins=True adds totals)
print()
print(pd.crosstab(
    schools["level"].map({1: "4-year", 2: "2-year"}),
    schools["control"].map({1: "public", 2: "private nonprofit"}),
    margins=True,
))

# Missingness AFTER filtering: compare to the earlier list
print()
print((schools.isna().mean() * 100).round(1).sort_values(ascending=False))

# Range checks: rates must be between 0 and 1, costs must be positive
print()
for col in ["admit_rate", "grad_rate_4yr", "grad_rate_2yr", "pct_international"]:
    bad = ((schools[col] < 0) | (schools[col] > 1)).sum()
    print(f"{col}: {bad} values outside 0-1")
print("cost_per_year <= 0:", (schools["cost_per_year"] <= 0).sum())

Start: 6273
After operating == 1: 6243
After keeping 2-year + 4-year: 4543
After dropping for-profits: 3635
After undergrads > 0: 3147

control  private nonprofit  public   All
level                                   
2-year                 125     820   945
4-year                1353     849  2202
All                   1478    1669  3147

grad_rate_2yr            70.6
sat_avg                  67.3
admit_rate               45.9
grad_rate_4yr            34.8
cost_per_year             9.7
median_earnings_10yr      7.6
tuition_out_of_state      6.6
tuition_in_state          6.6
open_admission_policy     3.1
lon                       0.0
lat                       0.0
name                      0.0
undergrads                0.0
pct_international         0.0
operating                 0.0
locale                    0.0
control                   0.0
main_degree               0.0
website                   0.0
state                     0.0
city                      0.0
level                     0.

In [4]:
# Step 3: build clean features for matching, then save.
import os

# 1. School type by the degree MOST students earn (PREDDEG), not the highest offered (ICLEVEL).
#    Catches community colleges that added a few bachelor's programs.
schools["school_type"] = schools["main_degree"].map({1: "2-year", 2: "2-year", 3: "4-year", 4: "4-year"})
print("ICLEVEL vs PREDDEG-based type:")
print(pd.crosstab(
    schools["level"].map({1: "4-year (ICLEVEL)", 2: "2-year (ICLEVEL)"}),
    schools["school_type"].fillna("unclassified"),
    margins=True,
))

# 2. Open admission flag: use the official policy field when present;
#    if it's blank for a school, fall back to "no admit rate reported".
policy = schools["open_admission_policy"]
schools["open_admission"] = np.where(policy.notna(), policy == 1, schools["admit_rate"].isna())
print("\nOpen admission schools:", schools["open_admission"].sum())
print("Missing admit_rate but NOT open admission:",
      (schools["admit_rate"].isna() & ~schools["open_admission"]).sum())

# 3. One graduation-rate column: 4-year rate for 4-year schools, 2-year rate otherwise.
#    Caveat: 2-year rates count early transfers as non-completers, so they understate JUCO success.
schools["grad_rate"] = np.where(
    schools["school_type"] == "4-year", schools["grad_rate_4yr"], schools["grad_rate_2yr"]
)


# 3b. cost_per_year (COSTT4_A) uses IN-STATE tuition for public schools.
#     International students pay out-of-state, so swap the tuition piece
#     to estimate what an international student is actually charged.
schools["cost_international"] = (
    schools["cost_per_year"] - schools["tuition_in_state"] + schools["tuition_out_of_state"]
)
pub = schools["control"] == 1
print("\nPublic schools, median cost: listed $%.0f vs international estimate $%.0f" % (
    schools.loc[pub, "cost_per_year"].median(), schools.loc[pub, "cost_international"].median()))

# 3c. City size label from the LOCALE code (first digit = type, second digit = size/distance)
locale_map = {
    11: "Large city", 12: "Midsize city", 13: "Small city",
    21: "Large suburb", 22: "Midsize suburb", 23: "Small suburb",
    31: "Town (fringe)", 32: "Town (distant)", 33: "Town (remote)",
    41: "Rural (fringe)", 42: "Rural (distant)", 43: "Rural (remote)",
}
schools["city_size"] = schools["locale"].map(locale_map)
print("\nSchools by city size:")
print(schools["city_size"].value_counts(dropna=False))

# 4. Drop columns we won't use: sat_avg (too sparse) and the two raw grad-rate columns (merged above)
schools = schools.drop(columns=["sat_avg", "grad_rate_4yr", "grad_rate_2yr"])

# 4b. Normalize website URLs: add https:// prefix where missing, keep NaN as NaN.
schools["website"] = schools["website"].apply(
    lambda u: ("https://" + str(u)) if pd.notna(u) and not str(u).startswith("http") else u
)
pct = schools["website"].notna().mean() * 100
print(f"\nSchools with a website: {pct:.1f}%")
print("5 example websites:")
print(schools["website"].dropna().head(5).to_string())

# 5. Final missingness check
print("\nMissing % after cleaning:")
print((schools.isna().mean() * 100).round(1).sort_values(ascending=False))

# 6. Save the cleaned dataset. Later notebooks load this instead of redoing all the steps above.
os.makedirs("../data/processed", exist_ok=True)
schools.to_csv("../data/processed/schools_clean.csv", index=False)
print("\nSaved:", schools.shape, "-> data/processed/schools_clean.csv")

ICLEVEL vs PREDDEG-based type:


school_type       2-year  4-year   All
level                                 
2-year (ICLEVEL)     945       0   945
4-year (ICLEVEL)     405    1797  2202
All                 1350    1797  3147

Open admission schools: 1428
Missing admit_rate but NOT open admission: 15

Public schools, median cost: listed $17725 vs international estimate $23395

Schools by city size:
city_size
Large city         644
Large suburb       583
Small city         422
Midsize city       342
Town (distant)     295
Rural (fringe)     221
Town (remote)      221
Rural (distant)    113
Midsize suburb      90
Rural (remote)      84
Town (fringe)       73
Small suburb        56
NaN                  3
Name: count, dtype: int64

Schools with a website: 100.0%
5 example websites:
0                 https://www.aamu.edu/
1                  https://www.uab.edu/
2    https://www.amridgeuniversity.edu/
3                  https://www.uah.edu/
4                https://www.alasu.edu/

Missing % after cleaning:
admit_rate     